In [ ]:
import scanpy as sc
import os
import scanpy as sc
import pandas as p
import numpy as np
from scipy import sparse
os.getcwd()

# with other unknown
整体流程（state_score 版本）

```text
第一步：生成 X_basic_celltype_proportion.csv 和 y_sample_label_numeric.csv
        ↓
中间这一步：生成 X_pseudobulk_marker.csv（原始 marker pseudobulk，保留作为原料）
        ↓
新增：用训练集 pseudobulk marker 计算 X_state_scores.csv
        ↓
再拼接生成 X_combined_basic_plus_state_scores.csv
        ↓
训练验证 LogisticRegression
        ↓
生成 unknown / new h5ad 的 state-score 特征并预测
```

注意：prediction data 的 state score 使用训练集 fit 出来的 gene-level mean/std 做 transform，不能对 prediction data 自己重新 fit。

In [ ]:
def get_mean_expression(sub_adata, genes):
    if len(genes) == 0:
        return pd.Series(dtype=float)
    sub = sub_adata[:, genes]
    X = sub.X
    mean_expr = np.asarray(X.mean(axis=0)).ravel() if sparse.issparse(X) else np.asarray(X.mean(axis=0)).ravel()
    return pd.Series(mean_expr, index=genes)

# Cell-type-specific plaque state marker panel.
# Pure lineage identity markers are excluded unless plaque single-cell or plaque-mechanism
# studies directly support their use as a disease-state marker within that cell type.
# Feature names are kept in human gene symbols; mouse data are mapped to these symbols before feature building.
plaque_state_marker_panel = {
    "Macrophage": [
        "CD163",
        "LGALS3", "APOE", "LPL", "TREM2", "SPP1",
        "IL1B", "TNF", "CCL2", "NLRP3",
        "MMP9", "MMP12", "CTSS", "CTSK"
    ],
    "Smooth muscle cell": [
        "ACTA2", "MYH11", "TAGLN", "CNN1", "MYOCD",
        "COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "FN1", "MGP",
        "RUNX2", "SOX9", "KLF4", "SPP1"
    ],
    "Endothelial cell": [
        "VCAM1", "ICAM1", "SELE", "PLVAP", "ANGPT2", "EDN1"
    ],
    "Fibroblast": [
        "LUM", "DCN", "COL1A1", "COL1A2", "COL3A1", "FN1",
        "MMP2", "MMP14", "TIMP3"
    ],
    "T cell": [
        "CCL5", "NKG7", "GZMB", "PRF1", "IFNG", "CXCR3"
    ]
}

# Explicit one-to-one mouse -> human symbols for the selected plaque_state_marker_panel.
# Avoid using gene.capitalize() here: that is a naming heuristic, not an ortholog table.
MOUSE_TO_HUMAN_ORTHOLOG = {
    "Cd163": "CD163", "Lgals3": "LGALS3", "Apoe": "APOE", "Lpl": "LPL",
    "Trem2": "TREM2", "Spp1": "SPP1", "Il1b": "IL1B", "Tnf": "TNF",
    "Ccl2": "CCL2", "Nlrp3": "NLRP3", "Mmp9": "MMP9", "Mmp12": "MMP12",
    "Ctss": "CTSS", "Ctsk": "CTSK", "Acta2": "ACTA2", "Myh11": "MYH11",
    "Tagln": "TAGLN", "Cnn1": "CNN1", "Myocd": "MYOCD", "Col1a1": "COL1A1",
    "Col1a2": "COL1A2", "Col3a1": "COL3A1", "Dcn": "DCN", "Lum": "LUM",
    "Fn1": "FN1", "Mgp": "MGP", "Runx2": "RUNX2", "Sox9": "SOX9",
    "Klf4": "KLF4", "Vcam1": "VCAM1", "Icam1": "ICAM1", "Sele": "SELE",
    "Plvap": "PLVAP", "Angpt2": "ANGPT2", "Edn1": "EDN1", "Mmp2": "MMP2",
    "Mmp14": "MMP14", "Timp3": "TIMP3", "Ccl5": "CCL5", "Nkg7": "NKG7",
    "Gzmb": "GZMB", "Prf1": "PRF1", "Ifng": "IFNG", "Cxcr3": "CXCR3"
}

marker_dict = plaque_state_marker_panel


def standardize_gene_names_for_marker_features(adata_in, mouse_to_human=None):
    """Map mouse-style marker symbols to human symbols so train/predict pseudobulk columns match."""
    if mouse_to_human is None:
        mouse_to_human = MOUSE_TO_HUMAN_ORTHOLOG
    old_names = pd.Index(adata_in.var_names.astype(str))
    new_names = pd.Index([mouse_to_human.get(g, g) for g in old_names])
    n_renamed = int((old_names != new_names).sum())
    if n_renamed > 0:
        print(f"Mapped {n_renamed} mouse-style marker gene names to human symbols for feature alignment.")
        adata_in.var_names = new_names.astype(str)
        adata_in.var_names_make_unique()
    return adata_in


def build_pseudobulk_features(adata_in, marker_dict, sample_col, celltype_col, include_ncells=False):
    all_genes = set(adata_in.var_names)
    filtered_marker_dict, missing_marker_report = {}, {}
    for ct, genes in marker_dict.items():
        present = [g for g in genes if g in all_genes]
        missing = [g for g in genes if g not in all_genes]
        filtered_marker_dict[ct] = present
        missing_marker_report[ct] = missing
    print("\n===== marker presence check =====")
    for ct in marker_dict:
        print(f"{ct}:")
        print("  present:", filtered_marker_dict[ct])
        print("  missing:", missing_marker_report[ct])
    filtered_marker_dict = {ct: genes for ct, genes in filtered_marker_dict.items() if len(genes) > 0}
    all_celltypes = set(adata_in.obs[celltype_col].astype(str).unique())
    print("\n===== cell types in adata =====")
    print(sorted(list(all_celltypes))[:50])
    valid_celltypes = [ct for ct in filtered_marker_dict if ct in all_celltypes]
    invalid_celltypes = [ct for ct in filtered_marker_dict if ct not in all_celltypes]
    if len(invalid_celltypes) > 0:
        print("\n这些 marker_dict 里的 cell type 不在 adata.obs 里，请检查命名：")
        print(invalid_celltypes)
    filtered_marker_dict = {ct: filtered_marker_dict[ct] for ct in valid_celltypes}
    if len(filtered_marker_dict) == 0:
        raise ValueError("没有任何有效的 cell type + marker 组合。")
    feature_rows = []
    sample_ids = adata_in.obs[sample_col].astype(str).unique().tolist()
    for sample_id in sample_ids:
        sample_mask = adata_in.obs[sample_col].astype(str) == str(sample_id)
        adata_sample = adata_in[sample_mask].copy()
        row_dict = {"sample": sample_id}
        for ct, genes in filtered_marker_dict.items():
            ct_mask = adata_sample.obs[celltype_col].astype(str) == ct
            n_cells = int(ct_mask.sum())
            if n_cells == 0:
                for g in genes:
                    row_dict[f"pb__{ct}__{g}"] = 0.0
                if include_ncells:
                    row_dict[f"pb_ncells__{ct}"] = 0
                continue
            adata_sub = adata_sample[ct_mask].copy()
            mean_expr = get_mean_expression(adata_sub, genes)
            for g in genes:
                row_dict[f"pb__{ct}__{g}"] = float(mean_expr[g])
            if include_ncells:
                row_dict[f"pb_ncells__{ct}"] = n_cells
        feature_rows.append(row_dict)
    X_pb = pd.DataFrame(feature_rows).set_index("sample").sort_index()
    return X_pb


In [ ]:
# =========================
# State score feature helpers
# =========================
# Keep X_pseudobulk_marker as the raw intermediate table, then compress marker genes
# into a smaller number of biologically grouped plaque-state scores.
# Pure lineage identity markers are excluded from STATE_MARKERS.

STATE_MARKERS = {
    "Macrophage_foam_lipid": {
        "celltype": "Macrophage",
        "genes": ["APOE", "LPL", "TREM2", "SPP1", "LGALS3"],
    },
    "Macrophage_inflammatory": {
        "celltype": "Macrophage",
        "genes": ["IL1B", "TNF", "CCL2", "NLRP3"],
    },
    "Macrophage_matrix_degradation": {
        "celltype": "Macrophage",
        "genes": ["MMP9", "MMP12", "CTSS", "CTSK"],
    },
    "Macrophage_CD163_angiogenic_permeability": {
        "celltype": "Macrophage",
        "genes": ["CD163"],
    },
    "SMC_contractile_state": {
        "celltype": "Smooth muscle cell",
        "genes": ["ACTA2", "MYH11", "TAGLN", "CNN1", "MYOCD"],
    },
    "SMC_synthetic_ECM_fibrous_cap": {
        "celltype": "Smooth muscle cell",
        "genes": ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "FN1", "MGP"],
    },
    "SMC_osteogenic_transition": {
        "celltype": "Smooth muscle cell",
        "genes": ["RUNX2", "SOX9", "KLF4", "SPP1"],
    },
    "Endothelial_activation_adhesion": {
        "celltype": "Endothelial cell",
        "genes": ["VCAM1", "ICAM1", "SELE"],
    },
    "Endothelial_angiogenesis_permeability": {
        "celltype": "Endothelial cell",
        "genes": ["PLVAP", "ANGPT2", "EDN1"],
    },
    "Fibroblast_ECM_remodeling": {
        "celltype": "Fibroblast",
        "genes": ["LUM", "DCN", "COL1A1", "COL1A2", "COL3A1", "FN1"],
    },
    "Fibroblast_matrix_degradation": {
        "celltype": "Fibroblast",
        "genes": ["MMP2", "MMP14"],
    },
    "Fibroblast_TIMP3_matrix_protection": {
        "celltype": "Fibroblast",
        "genes": ["TIMP3"],
    },
    "T_cell_cytotoxic_inflammatory": {
        "celltype": "T cell",
        "genes": ["CCL5", "NKG7", "GZMB", "PRF1", "IFNG", "CXCR3"],
    },
}

STATE_SCORE_MIN_GENES = 1

def _state_columns(x_pb, state_markers, min_genes=1):
    state_to_columns = {}
    report_rows = []

    for state_name, info in state_markers.items():
        celltype = info["celltype"]
        genes = info["genes"]
        columns = [f"pb__{celltype}__{gene}" for gene in genes]
        present = [col for col in columns if col in x_pb.columns]
        missing = [col for col in columns if col not in x_pb.columns]
        used = len(present) >= min_genes

        if used:
            state_to_columns[state_name] = present

        report_rows.append({
            "state": state_name,
            "celltype": celltype,
            "n_requested_genes": len(genes),
            "n_present_genes": len(present),
            "used": used,
            "present_columns": ";".join(present),
            "missing_columns": ";".join(missing),
        })

    return state_to_columns, report_rows


def fit_state_score_params(x_pb_train, state_markers=STATE_MARKERS, min_genes=STATE_SCORE_MIN_GENES):
    state_to_columns, report_rows = _state_columns(x_pb_train, state_markers, min_genes=min_genes)
    params = {}

    for state_name, columns in state_to_columns.items():
        mean = x_pb_train[columns].mean(axis=0)
        std = x_pb_train[columns].std(axis=0, ddof=0).replace(0, 1).fillna(1)
        params[state_name] = {
            "columns": columns,
            "mean": mean,
            "std": std,
        }

    return params, pd.DataFrame(report_rows)


def transform_state_scores(x_pb_data, params):
    scores = pd.DataFrame(index=x_pb_data.index)

    for state_name, state_params in params.items():
        columns = state_params["columns"]
        mean = state_params["mean"]
        std = state_params["std"]

        values = x_pb_data.reindex(columns=columns)
        values = values.fillna(mean)
        z_values = (values - mean) / std
        scores[f"score__{state_name}"] = z_values.mean(axis=1)

    return scores


def make_state_score_features(x_basic_part, x_pb_part, params):
    state_scores = transform_state_scores(x_pb_part, params)
    return pd.concat([x_basic_part, state_scores], axis=1)


def save_state_score_scaling_params(params, out_path):
    rows = []
    for state_name, state_params in params.items():
        for col in state_params["columns"]:
            rows.append({
                "state": state_name,
                "column": col,
                "mean": float(state_params["mean"].loc[col]),
                "std": float(state_params["std"].loc[col]),
            })
    pd.DataFrame(rows).to_csv(out_path, index=False)


## Plaque stability state-score rationale

这里不再把所有 pseudobulk marker gene 直接作为独立特征进入模型，而是先压缩成少量 plaque state scores：巨噬细胞吞噬/泡沫化/炎症/基质降解，SMC 收缩/ECM/骨化转分化，内皮稳态/激活血管新生，成纤维细胞 ECM 重塑和 T 细胞炎症毒性。这样能降低小样本高维过拟合风险，同时保留主要生物学状态信息。

## prepare input

In [ ]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse

# ========================= 参数 =========================
H5AD_PATH = "/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_allhuman/scPoli_concat_level3_marker_all_metadata.h5ad"
OUT_DIR = "./output_marker_state_score/"
os.makedirs(OUT_DIR, exist_ok=True)

SAMPLE_COL = "sample"
# CELLTYPE_COL = "cell_type_level1"
CELLTYPE_COL = "cell_type_level1_corrected"  # 应该用 cell_type_level1_corrected
LABEL_RAW_COL = "Plaque_type"
LABEL_COL = "Plaque_type_clean"

TRAIN_X_BASIC_PATH = f"{OUT_DIR}/X_basic_celltype_proportion.csv"
TRAIN_X_PB_PATH = f"{OUT_DIR}/X_pseudobulk_marker.csv"
TRAIN_X_STATE_PATH = f"{OUT_DIR}/X_state_scores.csv"
TRAIN_X_COMBINED_PATH = f"{OUT_DIR}/X_combined_basic_plus_state_scores.csv"
Y_LABEL_PATH = f"{OUT_DIR}/y_sample_label.csv"
Y_NUM_PATH = f"{OUT_DIR}/y_sample_label_numeric.csv"
STATE_MARKER_REPORT_PATH = f"{OUT_DIR}/state_score_marker_report.csv"
STATE_SCALING_PATH = f"{OUT_DIR}/state_score_scaling_params.csv"

OUT_X_BASIC_UNKNOWN = f"{OUT_DIR}/X_basic_unknown_celltype_proportion.csv"
OUT_X_PB_UNKNOWN = f"{OUT_DIR}/X_pseudobulk_marker_unknown.csv"
OUT_X_STATE_UNKNOWN = f"{OUT_DIR}/X_state_scores_unknown.csv"
OUT_X_UNKNOWN_COMBINED = f"{OUT_DIR}/X_unknown_combined_basic_plus_state_scores.csv"

marker_dict = plaque_state_marker_panel

# ========================= 1. 读取和预处理 h5ad =========================
print("Reading h5ad...")
adata = sc.read_h5ad(H5AD_PATH)
adata = adata[adata.obs[CELLTYPE_COL] != "unknown"].copy()
adata.var["original_feature_id"] = adata.var_names.astype(str)
adata.var_names = adata.var["original_gene_names"].astype(str)
adata.var_names_make_unique()
adata = standardize_gene_names_for_marker_features(adata)
adata.obs[LABEL_COL] = adata.obs[LABEL_RAW_COL].astype(str).str.strip().str.lower()

adata.obs[LABEL_COL] = adata.obs[LABEL_COL].replace({
    "nan": "unknown",
    "na": "unknown",
    "none": "unknown",
    "": "unknown",
    "unknown": "unknown",
    "Unknown": "unknown",
})
adata.obs[LABEL_COL] = adata.obs[LABEL_COL].fillna("unknown")

required_cols = [SAMPLE_COL, CELLTYPE_COL, LABEL_COL]
missing_cols = [c for c in required_cols if c not in adata.obs.columns]
if len(missing_cols) > 0:
    raise ValueError(f"adata.obs 缺少这些列: {missing_cols}")

print("adata shape after filtering unknown cell type:", adata.shape)
print("sample label distribution:")
print(adata.obs[LABEL_COL].value_counts(dropna=False))

# ========================= 2. 生成训练集 X_basic 和 y =========================
print("\nBuilding training X_basic and y...")
obs = adata.obs[[SAMPLE_COL, CELLTYPE_COL, LABEL_COL]].copy()
obs = obs.dropna(subset=[SAMPLE_COL, CELLTYPE_COL])
train_obs = obs[obs[LABEL_COL].isin(["stable", "unstable"])].copy()

print("训练细胞数:", train_obs.shape[0])
print("训练标签分布:")
print(train_obs[LABEL_COL].value_counts())

cell_count = pd.crosstab(train_obs[SAMPLE_COL], train_obs[CELLTYPE_COL])
cell_prop = cell_count.div(cell_count.sum(axis=1), axis=0)
cell_prop.columns = [f"prop__{c}" for c in cell_prop.columns]

sample_label = train_obs[[SAMPLE_COL, LABEL_COL]].drop_duplicates().groupby(SAMPLE_COL, observed=True)[LABEL_COL].agg(lambda x: x.iloc[0] if len(set(x)) == 1 else "conflict")
conflict_samples = sample_label[sample_label == "conflict"].index.tolist()
if len(conflict_samples) > 0:
    print("仍然冲突的 sample:", conflict_samples[:20])
    raise ValueError("还有 sample 存在多个标签，请进一步检查。")

sample_label = sample_label.rename("label")
common_samples = cell_prop.index.intersection(sample_label.index)
X_basic = cell_prop.loc[common_samples].copy().sort_index()
y = sample_label.loc[common_samples].copy().sort_index()
y_num = y.map({"stable": 0, "unstable": 1})

print("X_basic shape:", X_basic.shape)
print("y shape:", y_num.shape)
print("y label counts:")
print(y.value_counts())

X_basic.to_csv(TRAIN_X_BASIC_PATH)
y.to_csv(Y_LABEL_PATH, header=True)
y_num.to_csv(Y_NUM_PATH, header=True)
print(f"Saved: {TRAIN_X_BASIC_PATH}")
print(f"Saved: {Y_LABEL_PATH}")
print(f"Saved: {Y_NUM_PATH}")

# ========================= 3. 生成训练集 X_pseudobulk、X_state_scores 和 X_combined =========================
print("\nBuilding training X_pseudobulk...")
adata_train = adata[adata.obs[LABEL_COL].isin(["stable", "unstable"])].copy()
X_pb = build_pseudobulk_features(adata_train, marker_dict, SAMPLE_COL, CELLTYPE_COL)
X_pb.to_csv(TRAIN_X_PB_PATH)
print("X_pb shape:", X_pb.shape)
print(f"Saved: {TRAIN_X_PB_PATH}")

common_samples = X_basic.index.intersection(X_pb.index)
X_basic = X_basic.loc[common_samples]
X_pb = X_pb.loc[common_samples]
y_num = y_num.loc[common_samples]

state_params, marker_report = fit_state_score_params(X_pb, STATE_MARKERS, min_genes=STATE_SCORE_MIN_GENES)
X_state = transform_state_scores(X_pb, state_params)
X_combined = pd.concat([X_basic, X_state], axis=1)

marker_report.to_csv(STATE_MARKER_REPORT_PATH, index=False)
save_state_score_scaling_params(state_params, STATE_SCALING_PATH)
X_state.to_csv(TRAIN_X_STATE_PATH)
X_combined.to_csv(TRAIN_X_COMBINED_PATH)

print("State score marker report:")
print(marker_report[["state", "celltype", "n_present_genes", "used"]])
print("X_state shape:", X_state.shape)
print("X_combined state-score shape:", X_combined.shape)
print(f"Saved: {STATE_MARKER_REPORT_PATH}")
print(f"Saved: {STATE_SCALING_PATH}")
print(f"Saved: {TRAIN_X_STATE_PATH}")
print(f"Saved: {TRAIN_X_COMBINED_PATH}")

# ========================= 4. 生成 unknown 样本 X_basic_unknown =========================
print("\nBuilding unknown features...")
adata_unknown = adata[adata.obs[LABEL_COL].isin(["unknown"])].copy()
print("adata_unknown shape:", adata_unknown.shape)
print("unknown label counts:")
print(adata_unknown.obs[LABEL_COL].value_counts())

if adata_unknown.n_obs == 0:
    raise ValueError("没有找到标签为 unknown 的样本。")

obs_unknown = adata_unknown.obs[[SAMPLE_COL, CELLTYPE_COL, LABEL_COL]].copy()
obs_unknown = obs_unknown.dropna(subset=[SAMPLE_COL, CELLTYPE_COL])
cell_count_unknown = pd.crosstab(obs_unknown[SAMPLE_COL], obs_unknown[CELLTYPE_COL])
cell_prop_unknown = cell_count_unknown.div(cell_count_unknown.sum(axis=1), axis=0)
cell_prop_unknown.columns = [f"prop__{c}" for c in cell_prop_unknown.columns]

X_basic_unknown = cell_prop_unknown.copy()
X_basic_unknown = X_basic_unknown.reindex(columns=X_basic.columns, fill_value=0)
X_basic_unknown.to_csv(OUT_X_BASIC_UNKNOWN)
print("X_basic_unknown shape:", X_basic_unknown.shape)
print(f"Saved: {OUT_X_BASIC_UNKNOWN}")

# ========================= 5. 生成 unknown X_pseudobulk、X_state_scores 和 X_unknown_combined =========================
print("\nBuilding unknown X_pseudobulk...")
X_pb_unknown = build_pseudobulk_features(adata_unknown, marker_dict, SAMPLE_COL, CELLTYPE_COL)
X_pb_unknown = X_pb_unknown.reindex(columns=X_pb.columns, fill_value=0)
X_pb_unknown.to_csv(OUT_X_PB_UNKNOWN)
print("X_pb_unknown shape:", X_pb_unknown.shape)
print(f"Saved: {OUT_X_PB_UNKNOWN}")

common_unknown_samples = X_basic_unknown.index.intersection(X_pb_unknown.index)
X_basic_unknown = X_basic_unknown.loc[common_unknown_samples]
X_pb_unknown = X_pb_unknown.loc[common_unknown_samples]
X_state_unknown = transform_state_scores(X_pb_unknown, state_params)
X_unknown_combined = pd.concat([X_basic_unknown, X_state_unknown], axis=1)
X_unknown_combined = X_unknown_combined.reindex(columns=X_combined.columns, fill_value=0)

X_state_unknown.to_csv(OUT_X_STATE_UNKNOWN)
X_unknown_combined.to_csv(OUT_X_UNKNOWN_COMBINED)
print("X_state_unknown shape:", X_state_unknown.shape)
print("X_unknown_combined state-score shape:", X_unknown_combined.shape)
print(f"Saved: {OUT_X_STATE_UNKNOWN}")
print(f"Saved: {OUT_X_UNKNOWN_COMBINED}")

print("\nDone.")

# train

In [ ]:
# 这个是把有标签的数据全部拿来训练 没有标签的数据拿来预测-----没办法看预测准确性
# import pandas as pd
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LogisticRegression

# # 训练集
# X_train = pd.read_csv("./output_marker_state_score/X_combined_basic_plus_state_scores.csv", index_col=0)
# y_train = pd.read_csv("./output_marker_state_score/y_sample_label_numeric.csv", index_col=0).iloc[:, 0]

# # 训练最终模型
# final_model = Pipeline([
#     ("scaler", StandardScaler()),
#     ("clf", LogisticRegression(
#         penalty="l2",
#         C=1.0,
#         class_weight="balanced",
#         max_iter=1000,
#         random_state=42
#     ))
# ])

# final_model.fit(X_train, y_train)

# # unknown 特征
# X_unknown = pd.read_csv("./output_marker_state_score/X_unknown_combined_basic_plus_state_scores.csv", index_col=0)
# X_unknown = X_unknown.reindex(columns=X_train.columns, fill_value=0)

# # 预测 unstable 概率
# prob_unstable = final_model.predict_proba(X_unknown)[:, 1]

# result = pd.DataFrame({
#     "sample_id": X_unknown.index,
#     "prob_unstable": prob_unstable
# })

# # 按 0.6 阈值
# result["pred_label_0.6"] = (result["prob_unstable"] >= 0.6).astype(int)
# result["pred_label_0.6"] = result["pred_label_0.6"].map({
#     1: "unstable",
#     0: "non-unstable"
# })

# # 三档解释
# def assign_3class(p):
#     if p >= 0.8:
#         return "high-confidence unstable"
#     elif p >= 0.6:
#         return "unstable-like"
#     else:
#         return "non-unstable"

# result["pred_3class"] = result["prob_unstable"].apply(assign_3class)

# print(result.sort_values("prob_unstable", ascending=False))
# result.to_csv("./output_marker_state_score/unknown_prediction_final_lr_state_score.csv", index=False)

# 折线图-5次求平均

In [ ]:
# =========================
# State-score train-size CV helpers
# =========================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

STATE_SCORE_OUT_DIR = "./output_marker_state_score"
os.makedirs(STATE_SCORE_OUT_DIR, exist_ok=True)


def assign_3class(p, threshold=0.5, high_conf_threshold=0.8):
    if p >= high_conf_threshold:
        return "high-confidence unstable"
    if p >= threshold:
        return "unstable-like"
    return "non-unstable"


def build_lr_model(random_state=42):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="l2",
            C=1.0,
            class_weight="balanced",
            max_iter=1000,
            random_state=random_state,
        )),
    ])


def summarize_fold_metrics(fold_metrics_df):
    return (
        fold_metrics_df
        .groupby(["train_size", "train_percent", "n_splits"], as_index=False)
        .agg(
            mean_accuracy=("accuracy", "mean"),
            std_accuracy=("accuracy", "std"),
            mean_precision_unstable=("precision_unstable", "mean"),
            std_precision_unstable=("precision_unstable", "std"),
            mean_recall_unstable=("recall_unstable", "mean"),
            std_recall_unstable=("recall_unstable", "std"),
            mean_f1_unstable=("f1_unstable", "mean"),
            std_f1_unstable=("f1_unstable", "std"),
            mean_roc_auc=("roc_auc", "mean"),
            std_roc_auc=("roc_auc", "std"),
            mean_tn=("tn", "mean"),
            mean_fp=("fp", "mean"),
            mean_fn=("fn", "mean"),
            mean_tp=("tp", "mean"),
        )
    )


def run_state_score_train_size_cv(
    n_splits,
    threshold=0.5,
    random_state=42,
    summary_path=None,
    predictions_path=None,
):
    X_basic = pd.read_csv(f"{STATE_SCORE_OUT_DIR}/X_basic_celltype_proportion.csv", index_col=0)
    X_pb = pd.read_csv(f"{STATE_SCORE_OUT_DIR}/X_pseudobulk_marker.csv", index_col=0)
    y = pd.read_csv(f"{STATE_SCORE_OUT_DIR}/y_sample_label_numeric.csv", index_col=0).iloc[:, 0].astype(int)

    common_samples = X_basic.index.intersection(X_pb.index).intersection(y.index)
    X_basic = X_basic.loc[common_samples].sort_index()
    X_pb = X_pb.loc[common_samples].sort_index()
    y = y.loc[common_samples].sort_index()

    train_sizes = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
    summary_rows = []
    all_fold_predictions = []

    print("X_basic shape:", X_basic.shape)
    print("X_pb shape:", X_pb.shape)
    print("y distribution:")
    print(y.value_counts())

    for train_size in train_sizes:
        test_size = 1 - train_size
        print("\n" + "=" * 80)
        print(f"State-score CV | Training size = {int(train_size * 100)}%, Holdout size = {int(test_size * 100)}%")
        print("=" * 80)

        splitter = StratifiedShuffleSplit(
            n_splits=n_splits,
            train_size=train_size,
            test_size=test_size,
            random_state=random_state,
        )

        for fold_id, (train_idx, test_idx) in enumerate(splitter.split(X_basic, y), start=1):
            X_basic_train = X_basic.iloc[train_idx]
            X_basic_test = X_basic.iloc[test_idx]
            X_pb_train = X_pb.iloc[train_idx]
            X_pb_test = X_pb.iloc[test_idx]
            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            state_params, _ = fit_state_score_params(X_pb_train, STATE_MARKERS, min_genes=STATE_SCORE_MIN_GENES)
            X_train = make_state_score_features(X_basic_train, X_pb_train, state_params)
            X_test = make_state_score_features(X_basic_test, X_pb_test, state_params)
            X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

            model = build_lr_model(random_state=random_state)
            model.fit(X_train, y_train)

            prob_unstable = model.predict_proba(X_test)[:, 1]
            y_pred = (prob_unstable >= threshold).astype(int)
            cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
            auc = roc_auc_score(y_test, prob_unstable) if y_test.nunique() == 2 else np.nan

            summary_rows.append({
                "train_size": train_size,
                "train_percent": int(train_size * 100),
                "n_splits": n_splits,
                "fold": fold_id,
                "n_train": len(X_train),
                "n_test": len(X_test),
                "n_features": X_train.shape[1],
                "accuracy": accuracy_score(y_test, y_pred),
                "precision_unstable": precision_score(y_test, y_pred, zero_division=0),
                "recall_unstable": recall_score(y_test, y_pred, zero_division=0),
                "f1_unstable": f1_score(y_test, y_pred, zero_division=0),
                "roc_auc": auc,
                "tn": cm[0, 0],
                "fp": cm[0, 1],
                "fn": cm[1, 0],
                "tp": cm[1, 1],
            })

            fold_pred_df = pd.DataFrame({
                "train_size": train_size,
                "train_percent": int(train_size * 100),
                "n_splits": n_splits,
                "fold": fold_id,
                "sample_id": X_test.index,
                "true_label_numeric": y_test.values,
                "prob_unstable": prob_unstable,
                "pred_label_numeric": y_pred,
            })
            fold_pred_df["true_label"] = fold_pred_df["true_label_numeric"].map({1: "unstable", 0: "stable"})
            fold_pred_df["pred_label"] = fold_pred_df["pred_label_numeric"].map({1: "unstable", 0: "stable"})
            fold_pred_df["pred_3class"] = fold_pred_df["prob_unstable"].apply(lambda p: assign_3class(p, threshold=threshold))
            all_fold_predictions.append(fold_pred_df)

    fold_metrics_df = pd.DataFrame(summary_rows)
    summary_df = summarize_fold_metrics(fold_metrics_df).sort_values("train_size")
    all_predictions_df = pd.concat(all_fold_predictions, axis=0, ignore_index=True)

    print("\n" + "=" * 80)
    print("State-score cross-validation summary")
    print("=" * 80)
    print(summary_df)

    if summary_path is not None:
        summary_df.to_csv(summary_path, index=False)
        print("Saved:", summary_path)
    if predictions_path is not None:
        all_predictions_df.to_csv(predictions_path, index=False)
        print("Saved:", predictions_path)

    return summary_df, all_predictions_df


def plot_state_score_summary(summary_path, out_png=None, show_std=True):
    df = pd.read_csv(summary_path).sort_values("train_percent")
    x = df["train_percent"].to_numpy()
    series = [
        ("mean_accuracy", "std_accuracy", "Accuracy", "o"),
        ("mean_f1_unstable", "std_f1_unstable", "F1 Unstable", "s"),
        ("mean_roc_auc", "std_roc_auc", "ROC-AUC", "^"),
    ]

    plt.figure(figsize=(8, 5))
    std_arrays = []
    for mean_col, std_col, label, marker in series:
        mean_values = df[mean_col].to_numpy(dtype=float)
        plt.plot(
            x,
            mean_values,
            marker=marker,
            linewidth=2,
            label=f"Mean {label}",
        )
        if std_col in df.columns:
            std_arrays.append(df[std_col].fillna(0).to_numpy(dtype=float))

    if show_std and len(std_arrays) > 0:
        std_matrix = np.vstack(std_arrays)
        overall_std = np.sqrt(np.mean(np.square(std_matrix), axis=0))
        plt.plot(
            x,
            overall_std,
            marker="D",
            linestyle="--",
            linewidth=2,
            color="black",
            label="Overall STD",
        )

    plt.xlabel("Training set proportion (%)")
    plt.ylabel("Score / overall STD")
    plt.title("State-score LR metrics vs training proportion")
    plt.xticks(df["train_percent"])
    plt.ylim(0, 1)
    plt.grid(True, alpha=0.25)
    plt.legend(fontsize=9)
    plt.tight_layout()

    if out_png is not None:
        plt.savefig(out_png, dpi=600, bbox_inches="tight")
        print("Saved:", out_png)
    plt.show()


In [ ]:
summary_df, all_predictions_df = run_state_score_train_size_cv(
    n_splits=5,
    threshold=0.5,
    random_state=42,
    summary_path="./output_marker_state_score/lr_state_score_train_size_cv_summary_5.csv",
    predictions_path="./output_marker_state_score/lr_state_score_train_size_cv_all_fold_predictions_5.csv",
)

In [ ]:
plot_state_score_summary(
    "./output_marker_state_score/lr_state_score_train_size_cv_summary_5.csv",
    out_png="./output_marker_state_score/lr_state_score_train_size_cv_summary_5_with_std.png",
    show_std=True,
)

# 折线图---10次求平均

In [ ]:
summary_df, all_predictions_df = run_state_score_train_size_cv(
    n_splits=10,
    threshold=0.5,
    random_state=42,
    summary_path="./output_marker_state_score/lr_state_score_train_size_cv_summary_10.csv",
    predictions_path="./output_marker_state_score/lr_state_score_train_size_cv_all_fold_predictions_10.csv",
)

In [ ]:
plot_state_score_summary(
    "./output_marker_state_score/lr_state_score_train_size_cv_summary_10.csv",
    out_png="./output_marker_state_score/lr_state_score_train_size_cv_summary_10_with_std.png",
    show_std=True,
)

# 折线图----100次求平均

In [ ]:
summary_df, all_predictions_df = run_state_score_train_size_cv(
    n_splits=100,
    threshold=0.5,
    random_state=42,
    summary_path="./output_marker_state_score/lr_state_score_train_size_cv_summary_100.csv",
    predictions_path="./output_marker_state_score/lr_state_score_train_size_cv_all_fold_predictions_100.csv",
)

In [ ]:
plot_state_score_summary(
    "./output_marker_state_score/lr_state_score_train_size_cv_summary_100.csv",
    out_png="./output_marker_state_score/lr_state_score_train_size_cv_summary_100_with_std.png",
    show_std=True,
)

# 最终预测新数据 / unknown 样本

上面的交叉验证部分只是为了回答一个问题：**模型在不同训练集比例下稳不稳定**。  
真正要预测新数据时，不再留出测试集，而是：

```text
用所有已知 stable / unstable 样本训练最终模型
        ↓
把待预测样本整理成和训练集完全一样的特征列
        ↓
输出每个 sample 的 unstable 概率和预测类别
```

注意：85% 训练比例是用来评估模型稳定性的，不代表最终预测时只能用 85% 样本。最终预测通常用全部已标注样本训练模型。


### mouse

In [ ]:
# ============================================================
# 最终模型训练 + state-score 预测 unknown / 新 h5ad 数据
# ============================================================
# 1. PREDICT_MODE = "existing_unknown": 使用 prepare input 里已经生成的 unknown 特征。
# 2. PREDICT_MODE = "new_h5ad": 对一个新的 h5ad 重新生成 basic + pseudobulk，
#    然后使用训练集 fit 出来的 state-score mean/std 做 transform。
# ============================================================

import os
import pandas as pd
import numpy as np
import scanpy as sc

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix

# -------------------------
# 1. 参数设置
# -------------------------
PREDICT_MODE = "new_h5ad"
NEW_H5AD_PATH = "/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_mouse/scPoli_concat_level3_marker_allmouse.h5ad"
OUT_DIR = "./output_marker_state_score/mouse"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_FEATURE_DIR = "./output_marker_state_score"
PRED_THRESHOLD = 0.5
HIGH_CONF_THRESHOLD = 0.8

OUT_X_BASIC_PRED = f"{OUT_DIR}/X_basic_pred_celltype_proportion.csv"
OUT_X_PB_PRED = f"{OUT_DIR}/X_pseudobulk_marker_pred.csv"
OUT_X_STATE_PRED = f"{OUT_DIR}/X_state_scores_pred.csv"
OUT_X_COMBINED_PRED = f"{OUT_DIR}/X_pred_combined_basic_plus_state_scores.csv"
FINAL_PRED_OUT = f"{OUT_DIR}/final_prediction_unstable_probability_state_score.csv"
FINAL_COEF_OUT = f"{OUT_DIR}/final_lr_state_score_feature_coefficients.csv"

SAMPLE_COL = "sample"
# CELLTYPE_COL = "cell_type_level1"
CELLTYPE_COL = "cell_type_level1_corrected"
LABEL_RAW_COL = "Plaque_type"
LABEL_COL = "Plaque_type_clean"

marker_dict = plaque_state_marker_panel

# -------------------------
# 2. 读取训练原料特征和标签，fit state-score 参数
# -------------------------
X_basic_train = pd.read_csv(f"{TRAIN_FEATURE_DIR}/X_basic_celltype_proportion.csv", index_col=0)
X_pb_train = pd.read_csv(f"{TRAIN_FEATURE_DIR}/X_pseudobulk_marker.csv", index_col=0)
y_train = pd.read_csv(f"{TRAIN_FEATURE_DIR}/y_sample_label_numeric.csv", index_col=0).iloc[:, 0].astype(int)

common_train = X_basic_train.index.intersection(X_pb_train.index).intersection(y_train.index)
X_basic_train = X_basic_train.loc[common_train].sort_index()
X_pb_train = X_pb_train.loc[common_train].sort_index()
y_train = y_train.loc[common_train].sort_index()

state_params_final, marker_report_final = fit_state_score_params(
    X_pb_train,
    STATE_MARKERS,
    min_genes=STATE_SCORE_MIN_GENES,
)
X_train = make_state_score_features(X_basic_train, X_pb_train, state_params_final)

print("Training X shape:", X_train.shape)
print("Training y distribution:")
print(y_train.value_counts())

# -------------------------
# 3. 用全部已标注样本训练最终模型
# -------------------------
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l2",
        C=1.0,
        class_weight="balanced",
        max_iter=1000,
        random_state=42,
    )),
])

min_class_count = int(y_train.value_counts().min())
if y_train.nunique() == 2 and min_class_count >= 2:
    n_splits = min(5, min_class_count)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_prob = cross_val_predict(final_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    cv_pred = (cv_prob >= PRED_THRESHOLD).astype(int)
    print(f"\nSample-level CV ({n_splits}-fold):")
    print("  ROC-AUC:", round(roc_auc_score(y_train, cv_prob), 3))
    print("  balanced accuracy:", round(balanced_accuracy_score(y_train, cv_pred), 3))
    print("  confusion matrix [[stable, unstable] rows]:")
    print(confusion_matrix(y_train, cv_pred, labels=[0, 1]))
else:
    print("\nSkip CV: each class needs at least 2 samples.")

final_model.fit(X_train, y_train)
print("\nFinal model fitted using all labeled stable / unstable samples.")

# -------------------------
# 4A. 情况一：预测 prepare input 里已经生成好的 unknown 特征
# -------------------------
if PREDICT_MODE == "existing_unknown":
    X_basic_pred = pd.read_csv(f"{TRAIN_FEATURE_DIR}/X_basic_unknown_celltype_proportion.csv", index_col=0)
    X_pb_pred = pd.read_csv(f"{TRAIN_FEATURE_DIR}/X_pseudobulk_marker_unknown.csv", index_col=0)
    X_pb_pred = X_pb_pred.reindex(columns=X_pb_train.columns, fill_value=0)

    common_pred = X_basic_pred.index.intersection(X_pb_pred.index)
    X_basic_pred = X_basic_pred.loc[common_pred].reindex(columns=X_basic_train.columns, fill_value=0)
    X_pb_pred = X_pb_pred.loc[common_pred]
    pred_source = "existing_unknown"

# -------------------------
# 4B. 情况二：预测一个新的 h5ad 文件
# -------------------------
elif PREDICT_MODE == "new_h5ad":
    if not os.path.exists(NEW_H5AD_PATH):
        raise FileNotFoundError(
            f"找不到 NEW_H5AD_PATH: {NEW_H5AD_PATH}\n"
            "请把 NEW_H5AD_PATH 改成你的新 h5ad 文件路径。"
        )

    print("\nReading new h5ad...")
    adata_new = sc.read_h5ad(NEW_H5AD_PATH)

    if CELLTYPE_COL not in adata_new.obs.columns:
        raise ValueError(f"新数据 adata.obs 里缺少 CELLTYPE_COL: {CELLTYPE_COL}")
    if SAMPLE_COL not in adata_new.obs.columns:
        raise ValueError(f"新数据 adata.obs 里缺少 SAMPLE_COL: {SAMPLE_COL}")

    adata_new = adata_new[adata_new.obs[CELLTYPE_COL] != "unknown"].copy()

    if "original_gene_names" in adata_new.var.columns:
        adata_new.var["original_feature_id"] = adata_new.var_names.astype(str)
        adata_new.var_names = adata_new.var["original_gene_names"].astype(str)
        adata_new.var_names_make_unique()

    adata_new = standardize_gene_names_for_marker_features(adata_new)
    print("New adata shape after filtering unknown cell type:", adata_new.shape)

    obs_new = adata_new.obs[[SAMPLE_COL, CELLTYPE_COL]].copy()
    obs_new = obs_new.dropna(subset=[SAMPLE_COL, CELLTYPE_COL])

    cell_count_new = pd.crosstab(obs_new[SAMPLE_COL], obs_new[CELLTYPE_COL])
    cell_prop_new = cell_count_new.div(cell_count_new.sum(axis=1), axis=0)
    cell_prop_new.columns = [f"prop__{c}" for c in cell_prop_new.columns]
    X_basic_pred = cell_prop_new.reindex(columns=X_basic_train.columns, fill_value=0)

    X_pb_pred = build_pseudobulk_features(
        adata_new,
        marker_dict,
        SAMPLE_COL,
        CELLTYPE_COL,
    )
    X_pb_pred = X_pb_pred.reindex(columns=X_pb_train.columns, fill_value=0)

    common_pred = X_basic_pred.index.intersection(X_pb_pred.index)
    X_basic_pred = X_basic_pred.loc[common_pred]
    X_pb_pred = X_pb_pred.loc[common_pred]
    pred_source = "new_h5ad"

else:
    raise ValueError('PREDICT_MODE 只能是 "existing_unknown" 或 "new_h5ad"')

# -------------------------
# 4C. 用训练集 state-score 参数 transform prediction data
# -------------------------
X_state_pred = transform_state_scores(X_pb_pred, state_params_final)
X_pred = pd.concat([X_basic_pred, X_state_pred], axis=1)
X_pred = X_pred.reindex(columns=X_train.columns, fill_value=0)

X_basic_pred.to_csv(OUT_X_BASIC_PRED)
X_pb_pred.to_csv(OUT_X_PB_PRED)
X_state_pred.to_csv(OUT_X_STATE_PRED)
X_pred.to_csv(OUT_X_COMBINED_PRED)

print("\nPrediction X shape:", X_pred.shape)
print("Prediction source:", pred_source)
print(f"Saved prediction features: {OUT_X_COMBINED_PRED}")

# -------------------------
# 5. 预测 unstable 概率
# -------------------------
prob_unstable = final_model.predict_proba(X_pred)[:, 1]

pred_df = pd.DataFrame({
    "sample_id": X_pred.index,
    "prob_unstable": prob_unstable,
})

pred_df[f"pred_label_threshold_{PRED_THRESHOLD}"] = np.where(
    pred_df["prob_unstable"] >= PRED_THRESHOLD,
    "unstable-like",
    "non-unstable",
)

pred_df["pred_3class"] = pred_df["prob_unstable"].apply(
    lambda p: assign_3class(p, threshold=PRED_THRESHOLD, high_conf_threshold=HIGH_CONF_THRESHOLD)
)
pred_df = pred_df.sort_values("prob_unstable", ascending=False)

print("\nPrediction result:")
print(pred_df)

pred_df.to_csv(FINAL_PRED_OUT, index=False)
print(f"\nSaved prediction result: {FINAL_PRED_OUT}")

# -------------------------
# 6. 导出 LogisticRegression 特征系数
# -------------------------
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coef": final_model.named_steps["clf"].coef_[0],
})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)
coef_df.to_csv(FINAL_COEF_OUT, index=False)
print(f"Saved feature coefficients: {FINAL_COEF_OUT}")

print("\nTop positive features: higher value pushes prediction toward unstable")
print(coef_df.sort_values("coef", ascending=False).head(20))

print("\nTop negative features: higher value pushes prediction toward non-unstable")
print(coef_df.sort_values("coef", ascending=True).head(20))

In [ ]:
adata_new

In [ ]:
len(adata_new.obs['sample'].value_counts())

In [ ]:
import pandas as pd

pred_path = "./output_marker_state_score/mouse/final_prediction_unstable_probability_state_score.csv"
pred = pd.read_csv(pred_path)
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_mouse/scPoli_concat_level3_marker_allmouse.h5ad")

pred_sub = pred[[
    "sample_id",
    "pred_label_threshold_0.5",
    "pred_3class"
]].rename(columns={
    "pred_label_threshold_0.5": "Plaque_type_pred",
    "pred_3class": "Plaque_type_pred_3class"
})

mapping_1 = pred_sub.set_index("sample_id")["Plaque_type_pred"]
mapping_2 = pred_sub.set_index("sample_id")["Plaque_type_pred_3class"]

adata.obs["Plaque_type_pred"] = adata.obs["sample"].map(mapping_1)
adata.obs["Plaque_type_pred_3class"] = adata.obs["sample"].map(mapping_2)

In [ ]:
adata.obs['Plaque_type_pred'].value_counts()

In [ ]:
### 下一步转换成 rds 文件
adata.write("./mouse_plaque_type_pred_state_score.h5ad")
adata

### human

In [ ]:
# ============================================================
# 最终模型训练 + state-score 预测 unknown / 新 h5ad 数据
# ============================================================
# 1. PREDICT_MODE = "existing_unknown": 使用 prepare input 里已经生成的 unknown 特征。
# 2. PREDICT_MODE = "new_h5ad": 对一个新的 h5ad 重新生成 basic + pseudobulk，
#    然后使用训练集 fit 出来的 state-score mean/std 做 transform。
# ============================================================

import os
import pandas as pd
import numpy as np
import scanpy as sc

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix

# -------------------------
# 1. 参数设置
# -------------------------
PREDICT_MODE = "existing_unknown"
NEW_H5AD_PATH = "/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_mouse/scPoli_concat_level3_marker_allmouse.h5ad"
OUT_DIR = "./output_marker_state_score"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_FEATURE_DIR = "./output_marker_state_score"
PRED_THRESHOLD = 0.5
HIGH_CONF_THRESHOLD = 0.8

OUT_X_BASIC_PRED = f"{OUT_DIR}/X_basic_pred_celltype_proportion.csv"
OUT_X_PB_PRED = f"{OUT_DIR}/X_pseudobulk_marker_pred.csv"
OUT_X_STATE_PRED = f"{OUT_DIR}/X_state_scores_pred.csv"
OUT_X_COMBINED_PRED = f"{OUT_DIR}/X_pred_combined_basic_plus_state_scores.csv"
FINAL_PRED_OUT = f"{OUT_DIR}/final_prediction_unstable_probability_state_score.csv"
FINAL_COEF_OUT = f"{OUT_DIR}/final_lr_state_score_feature_coefficients.csv"

SAMPLE_COL = "sample"
# CELLTYPE_COL = "cell_type_level1"
CELLTYPE_COL = "cell_type_level1_corrected"
LABEL_RAW_COL = "Plaque_type"
LABEL_COL = "Plaque_type_clean"

marker_dict = plaque_state_marker_panel

# -------------------------
# 2. 读取训练原料特征和标签，fit state-score 参数
# -------------------------
X_basic_train = pd.read_csv(f"{TRAIN_FEATURE_DIR}/X_basic_celltype_proportion.csv", index_col=0)
X_pb_train = pd.read_csv(f"{TRAIN_FEATURE_DIR}/X_pseudobulk_marker.csv", index_col=0)
y_train = pd.read_csv(f"{TRAIN_FEATURE_DIR}/y_sample_label_numeric.csv", index_col=0).iloc[:, 0].astype(int)

common_train = X_basic_train.index.intersection(X_pb_train.index).intersection(y_train.index)
X_basic_train = X_basic_train.loc[common_train].sort_index()
X_pb_train = X_pb_train.loc[common_train].sort_index()
y_train = y_train.loc[common_train].sort_index()

state_params_final, marker_report_final = fit_state_score_params(
    X_pb_train,
    STATE_MARKERS,
    min_genes=STATE_SCORE_MIN_GENES,
)
X_train = make_state_score_features(X_basic_train, X_pb_train, state_params_final)

print("Training X shape:", X_train.shape)
print("Training y distribution:")
print(y_train.value_counts())

# -------------------------
# 3. 用全部已标注样本训练最终模型
# -------------------------
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l2",
        C=1.0,
        class_weight="balanced",
        max_iter=1000,
        random_state=42,
    )),
])

min_class_count = int(y_train.value_counts().min())
if y_train.nunique() == 2 and min_class_count >= 2:
    n_splits = min(5, min_class_count)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_prob = cross_val_predict(final_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    cv_pred = (cv_prob >= PRED_THRESHOLD).astype(int)
    print(f"\nSample-level CV ({n_splits}-fold):")
    print("  ROC-AUC:", round(roc_auc_score(y_train, cv_prob), 3))
    print("  balanced accuracy:", round(balanced_accuracy_score(y_train, cv_pred), 3))
    print("  confusion matrix [[stable, unstable] rows]:")
    print(confusion_matrix(y_train, cv_pred, labels=[0, 1]))
else:
    print("\nSkip CV: each class needs at least 2 samples.")

final_model.fit(X_train, y_train)
print("\nFinal model fitted using all labeled stable / unstable samples.")

# -------------------------
# 4A. 情况一：预测 prepare input 里已经生成好的 unknown 特征
# -------------------------
if PREDICT_MODE == "existing_unknown":
    X_basic_pred = pd.read_csv(f"{TRAIN_FEATURE_DIR}/X_basic_unknown_celltype_proportion.csv", index_col=0)
    X_pb_pred = pd.read_csv(f"{TRAIN_FEATURE_DIR}/X_pseudobulk_marker_unknown.csv", index_col=0)
    X_pb_pred = X_pb_pred.reindex(columns=X_pb_train.columns, fill_value=0)

    common_pred = X_basic_pred.index.intersection(X_pb_pred.index)
    X_basic_pred = X_basic_pred.loc[common_pred].reindex(columns=X_basic_train.columns, fill_value=0)
    X_pb_pred = X_pb_pred.loc[common_pred]
    pred_source = "existing_unknown"

# -------------------------
# 4B. 情况二：预测一个新的 h5ad 文件
# -------------------------
elif PREDICT_MODE == "new_h5ad":
    if not os.path.exists(NEW_H5AD_PATH):
        raise FileNotFoundError(
            f"找不到 NEW_H5AD_PATH: {NEW_H5AD_PATH}\n"
            "请把 NEW_H5AD_PATH 改成你的新 h5ad 文件路径。"
        )

    print("\nReading new h5ad...")
    adata_new = sc.read_h5ad(NEW_H5AD_PATH)

    if CELLTYPE_COL not in adata_new.obs.columns:
        raise ValueError(f"新数据 adata.obs 里缺少 CELLTYPE_COL: {CELLTYPE_COL}")
    if SAMPLE_COL not in adata_new.obs.columns:
        raise ValueError(f"新数据 adata.obs 里缺少 SAMPLE_COL: {SAMPLE_COL}")

    adata_new = adata_new[adata_new.obs[CELLTYPE_COL] != "unknown"].copy()

    if "original_gene_names" in adata_new.var.columns:
        adata_new.var["original_feature_id"] = adata_new.var_names.astype(str)
        adata_new.var_names = adata_new.var["original_gene_names"].astype(str)
        adata_new.var_names_make_unique()

    adata_new = standardize_gene_names_for_marker_features(adata_new)
    print("New adata shape after filtering unknown cell type:", adata_new.shape)

    obs_new = adata_new.obs[[SAMPLE_COL, CELLTYPE_COL]].copy()
    obs_new = obs_new.dropna(subset=[SAMPLE_COL, CELLTYPE_COL])

    cell_count_new = pd.crosstab(obs_new[SAMPLE_COL], obs_new[CELLTYPE_COL])
    cell_prop_new = cell_count_new.div(cell_count_new.sum(axis=1), axis=0)
    cell_prop_new.columns = [f"prop__{c}" for c in cell_prop_new.columns]
    X_basic_pred = cell_prop_new.reindex(columns=X_basic_train.columns, fill_value=0)

    X_pb_pred = build_pseudobulk_features(
        adata_new,
        marker_dict,
        SAMPLE_COL,
        CELLTYPE_COL,
    )
    X_pb_pred = X_pb_pred.reindex(columns=X_pb_train.columns, fill_value=0)

    common_pred = X_basic_pred.index.intersection(X_pb_pred.index)
    X_basic_pred = X_basic_pred.loc[common_pred]
    X_pb_pred = X_pb_pred.loc[common_pred]
    pred_source = "new_h5ad"

else:
    raise ValueError('PREDICT_MODE 只能是 "existing_unknown" 或 "new_h5ad"')

# -------------------------
# 4C. 用训练集 state-score 参数 transform prediction data
# -------------------------
X_state_pred = transform_state_scores(X_pb_pred, state_params_final)
X_pred = pd.concat([X_basic_pred, X_state_pred], axis=1)
X_pred = X_pred.reindex(columns=X_train.columns, fill_value=0)

X_basic_pred.to_csv(OUT_X_BASIC_PRED)
X_pb_pred.to_csv(OUT_X_PB_PRED)
X_state_pred.to_csv(OUT_X_STATE_PRED)
X_pred.to_csv(OUT_X_COMBINED_PRED)

print("\nPrediction X shape:", X_pred.shape)
print("Prediction source:", pred_source)
print(f"Saved prediction features: {OUT_X_COMBINED_PRED}")

# -------------------------
# 5. 预测 unstable 概率
# -------------------------
prob_unstable = final_model.predict_proba(X_pred)[:, 1]

pred_df = pd.DataFrame({
    "sample_id": X_pred.index,
    "prob_unstable": prob_unstable,
})

pred_df[f"pred_label_threshold_{PRED_THRESHOLD}"] = np.where(
    pred_df["prob_unstable"] >= PRED_THRESHOLD,
    "unstable-like",
    "non-unstable",
)

pred_df["pred_3class"] = pred_df["prob_unstable"].apply(
    lambda p: assign_3class(p, threshold=PRED_THRESHOLD, high_conf_threshold=HIGH_CONF_THRESHOLD)
)
pred_df = pred_df.sort_values("prob_unstable", ascending=False)

print("\nPrediction result:")
print(pred_df)

pred_df.to_csv(FINAL_PRED_OUT, index=False)
print(f"\nSaved prediction result: {FINAL_PRED_OUT}")

# -------------------------
# 6. 导出 LogisticRegression 特征系数
# -------------------------
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coef": final_model.named_steps["clf"].coef_[0],
})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)
coef_df.to_csv(FINAL_COEF_OUT, index=False)
print(f"Saved feature coefficients: {FINAL_COEF_OUT}")

print("\nTop positive features: higher value pushes prediction toward unstable")
print(coef_df.sort_values("coef", ascending=False).head(20))

print("\nTop negative features: higher value pushes prediction toward non-unstable")
print(coef_df.sort_values("coef", ascending=True).head(20))

In [ ]:
import pandas as pd

pred_path = "./output_marker_state_score/final_prediction_unstable_probability_state_score.csv"
pred = pd.read_csv(pred_path)
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_allhuman/scPoli_concat_level3_marker_all_metadata.h5ad")

pred_sub = pred[[
    "sample_id",
    "pred_label_threshold_0.5",
    "pred_3class"
]].rename(columns={
    "pred_label_threshold_0.5": "Plaque_type_pred",
    "pred_3class": "Plaque_type_pred_3class"
})

mapping_1 = pred_sub.set_index("sample_id")["Plaque_type_pred"]
mapping_2 = pred_sub.set_index("sample_id")["Plaque_type_pred_3class"]

adata.obs["Plaque_type_pred"] = adata.obs["sample"].map(mapping_1)
adata.obs["Plaque_type_pred_3class"] = adata.obs["sample"].map(mapping_2)

In [ ]:
adata.obs['Plaque_type_pred'].value_counts()

In [ ]:
adata.write("./human_plaque_type_pred_state_score.h5ad")

#### 富集分析

用R试一下